In [ ]:
import json
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp
sys.path.insert(0, str(hp.HODDIES_ROOT))
for module_name in list(sys.modules):
    if module_name == 'HODDIES' or module_name.startswith('HODDIES.'):
        del sys.modules[module_name]

import numpy as np
import matplotlib.pyplot as plt
from HODDIES import HOD
from abacus_reference_cosmology import SERIALIZABLE_PARAMETERS, PARAMETERS
import HODDIES

print(f'Using HODDIES from {HODDIES.__file__}')


In [ ]:
from cosmoprimo.fiducial import Cosmology

abacus_cosmo = Cosmology(
    Omega0_m=SERIALIZABLE_PARAMETERS['Omega_m'],
    Omega_b=SERIALIZABLE_PARAMETERS['Omega_b'],
    h=SERIALIZABLE_PARAMETERS['h'],
    sigma8=SERIALIZABLE_PARAMETERS['sigma8'],
    n_s=SERIALIZABLE_PARAMETERS['n_s'],
    w0_fld=SERIALIZABLE_PARAMETERS['w0'],
    wa_fld=SERIALIZABLE_PARAMETERS['wa'],
    T_cmb=SERIALIZABLE_PARAMETERS['Tcmb0_K'],
    N_eff=SERIALIZABLE_PARAMETERS['Neff'],
    m_ncdm=[],
    engine='class',
)

print(SERIALIZABLE_PARAMETERS)
abacus_cosmo.get_params()


In [ ]:
target, model, realization = 'Abacus_Small', 'nDGP_N1.2', 's1234'
boxsize = hp.TARGETS[target]['boxsize']
redshift = 0.500

OUTPUT_DIR = hp.halo_catalog_dir(target, model, realization, redshift)
HALO_CATALOG = OUTPUT_DIR / 'halos200c_catalog.npy'
HALO_METADATA = Path(str(HALO_CATALOG) + '.json')
HODDIES_CATALOG = OUTPUT_DIR / 'halos200c_hoddies.npy'
HMF_OUTPUT = OUTPUT_DIR / 'halos200c_hmf.txt'
ANALYSIS_DIR = OUTPUT_DIR / 'analysis'
PARAM_FILE = str(hp.NOTEBOOK_DIR / f'HOD_parameter_file_z{redshift:.3f}.yaml')

print(f'Halo catalog: {HALO_CATALOG}')
print(f'HODDIES catalog: {HODDIES_CATALOG}')
print(f'HMF output: {HMF_OUTPUT}')
print(f'Analysis directory: {ANALYSIS_DIR}')


In [ ]:
if not HODDIES_CATALOG.exists():
    raise FileNotFoundError(
        f'Missing {HODDIES_CATALOG}. Run jobs/halo_finder.sh on this run first, '
        'then come back to this notebook.'
    )

arr = np.load(HODDIES_CATALOG)
print(f'Loaded {len(arr):,} HODDIES-ready halos')
print(arr.dtype)

if HALO_METADATA.exists():
    with HALO_METADATA.open() as f:
        halo_metadata = json.load(f)
    print(json.dumps(halo_metadata, indent=2))
else:
    halo_metadata = {}

expected_cosmology = SERIALIZABLE_PARAMETERS
halo_cosmology = halo_metadata.get('cosmology')
if halo_cosmology is not None:
    mismatches = {
        key: (halo_cosmology.get(key), expected)
        for key, expected in expected_cosmology.items()
        if key in halo_cosmology and not np.isclose(float(halo_cosmology[key]), float(expected))
    }
    if mismatches:
        raise ValueError(f'Halo catalog cosmology metadata does not match AbacusSummit c000 reference: {mismatches}')
else:
    print('Halo catalog metadata has no cosmology block; using notebook AbacusSummit c000 reference cosmology.')


In [ ]:
print('Mass range:', arr['Mh'].min(), arr['Mh'].max())
print('log10_Mh range:', arr['log10_Mh'].min(), arr['log10_Mh'].max())
print('Position mins:', np.min(np.vstack([arr['x'], arr['y'], arr['z']]).T, axis=0))
print('Position maxs:', np.max(np.vstack([arr['x'], arr['y'], arr['z']]).T, axis=0))


In [ ]:
HOD_obj = HOD(hcat_file=arr, **{'hcat': {'boxsize': boxsize}}, param_file=PARAM_FILE)
HOD_obj.args['hcat']['z_simu'] = redshift
HOD_obj.args['cosmo'].update({
    'fiducial_cosmo': None,
    'engine': 'class',
    'h': SERIALIZABLE_PARAMETERS['h'],
    'Omega_m': SERIALIZABLE_PARAMETERS['Omega_m'],
    'Omega_cdm': SERIALIZABLE_PARAMETERS['Omega_cdm'],
    'Omega_L': SERIALIZABLE_PARAMETERS['Omega_Lambda'],
    'Omega_b': SERIALIZABLE_PARAMETERS['Omega_b'],
    'sigma_8': SERIALIZABLE_PARAMETERS['sigma8'],
    'n_s': SERIALIZABLE_PARAMETERS['n_s'],
    'w0_fdl': SERIALIZABLE_PARAMETERS['w0'],
    'wa_fdl': SERIALIZABLE_PARAMETERS['wa'],
})
HOD_obj.cosmo = abacus_cosmo


In [ ]:
HOD_obj.plot_initial_HMF()

In [ ]:
cat = HOD_obj.make_mock_cat(tracers='LRG', fix_seed=42)

In [ ]:
HOD_obj.plot_HMF(cat,tracer='LRG')

In [ ]:
HOD_obj.plot_HMF(cat, show_sat=True, tracer='LRG')

In [ ]:
HOD_obj.HOD_plot()

## Styled plots (power_spectrum2 style)

In [ ]:
FIG_SIZE = (6.6, 4.6)
LABEL_SIZE = 14.5
TICK_SIZE = 12.5
LEGEND_SIZE = 11.5
LINE_WIDTH = 1.9

COLA_COLOR = '#c47f2b'   # orange
BART_COLOR = '#1f4e79'   # navy
RATIO_COLOR = '#2b7a78'  # teal
REF_COLOR = '0.35'       # gray reference lines


def style_axis(ax, tick_size=TICK_SIZE):
    ax.minorticks_on()
    ax.grid(True, which='major', ls=':', color='0.82', alpha=0.75, lw=0.5)
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True,
                    labelsize=tick_size, length=4.6, width=0.95)
    ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True,
                    labelsize=tick_size - 1.2, length=2.8, width=0.75)
    for spine in ax.spines.values():
        spine.set_linewidth(0.95)

In [ ]:
def plot_hmf_styled(cat, tracer='LRG', show_sat=False, mass_range=(10.8, 15), bins=100):
    mask = cat['TRACER'] == tracer
    data = cat['log10_Mh'][mask]

    fig, ax = plt.subplots(figsize=FIG_SIZE, dpi=300, constrained_layout=True)

    ax.hist(data, histtype='step', bins=bins, range=mass_range,
            color=BART_COLOR, lw=LINE_WIDTH, label=tracer)

    if show_sat:
        ax.hist(cat['log10_Mh'][mask & (cat['Central'] == 1)], histtype='step', bins=bins,
                range=mass_range, color=COLA_COLOR, lw=LINE_WIDTH, ls='--', label='Centrals')
        ax.hist(cat['log10_Mh'][mask & (cat['Central'] == 0)], histtype='step', bins=bins,
                range=mass_range, color=RATIO_COLOR, lw=LINE_WIDTH, ls=':', label='Satellites')

    ax.set_yscale('log')
    ax.set_xlabel(r'$\log(M_h\,[M_{\odot}])$', fontsize=LABEL_SIZE)
    ax.set_ylabel(r'd$N$/d$M_h$ [$h$/Mpc$^3$]', fontsize=LABEL_SIZE)
    ax.legend(fontsize=LEGEND_SIZE, frameon=False, handlelength=1.8, borderaxespad=0.35, labelspacing=0.35)

    pad = 0.03 * (data.max() - data.min())
    ax.set_xlim(data.min() - pad, data.max() + pad)

    style_axis(ax)
    plt.show()


plot_hmf_styled(cat, tracer='LRG')

In [ ]:
plot_hmf_styled(cat, tracer='LRG', show_sat=True)

In [ ]:
def plot_hod_styled(HOD_obj, tracer='LRG', logM=np.linspace(10.8, 15, 100)):
    pc, ps, _ = HOD_obj._HOD__init_hod_param(tracer)
    cen = np.asarray([HOD_obj._fun_cHOD[tracer](lM, pc) for lM in logM])
    sat = np.nan_to_num([HOD_obj._fun_sHOD[tracer](lM, ps) for lM in logM])

    fig, ax = plt.subplots(figsize=FIG_SIZE, dpi=300, constrained_layout=True)

    ylim_low, ylim_high = 0.001, 10
    ax.axhline(y=1, ls='--', c=REF_COLOR, lw=1.1)
    ax.plot(logM, cen, lw=LINE_WIDTH, color=COLA_COLOR, label='Centrals')
    ax.plot(logM, sat, lw=LINE_WIDTH, ls='--', color=BART_COLOR, label='Satellites')

    ax.set_yscale('log')
    ax.set_ylim(ylim_low, ylim_high)
    ax.set_xlabel(r'$\log(M_h\,[M_{\odot}])$', fontsize=LABEL_SIZE)
    ax.set_ylabel(r'$\langle N_{\mathrm{gal}} \rangle$', fontsize=LABEL_SIZE)
    ax.legend(fontsize=LEGEND_SIZE, frameon=False, handlelength=1.8, borderaxespad=0.35, labelspacing=0.35)

    visible = (cen > ylim_low) | (sat > ylim_low)
    xmin = logM[visible].min() if visible.any() else logM.min()
    pad = 0.03 * (logM.max() - xmin)
    ax.set_xlim(xmin - pad, logM.max() + pad)

    style_axis(ax)
    plt.show()


plot_hod_styled(HOD_obj, tracer='LRG')

In [ ]:
rp, wp = HOD_obj.get_wp(cat, tracers='LRG')
s, xi = HOD_obj.get_2PCF(cat, tracers='LRG')

In [ ]:
HOD_obj.args['LRG']['link_sat_to_central'] = True

In [ ]:
plt.plot(rp,rp*wp)
plt.xscale('log')
plt.xlabel('$r_p$ [$\mathrm{{Mpc}}/h$]')
plt.ylabel(r'$r_p \cdot w_p(r_p)$ [$\mathrm{{Mpc}}/h$]$^2$')
plt.show()

In [ ]:
def plot_wp_styled(rp, wp, label='LRG'):
    fig, ax = plt.subplots(figsize=FIG_SIZE, dpi=300, constrained_layout=True)
    ax.semilogx(rp, rp * wp, lw=LINE_WIDTH, color=BART_COLOR, label=label)
    ax.set_xlabel(r'$r_p\,[\mathrm{Mpc}/h]$', fontsize=LABEL_SIZE)
    ax.set_ylabel(r'$r_p \cdot w_p(r_p)\,[\mathrm{Mpc}/h]^2$', fontsize=LABEL_SIZE)
    ax.legend(fontsize=LEGEND_SIZE, frameon=False, handlelength=1.8, borderaxespad=0.35, labelspacing=0.35)
    style_axis(ax)
    plt.show()


plot_wp_styled(rp, wp, label='LRG')

In [ ]:
plt.plot(s,s*xi[0], label='l=0')
plt.plot(s,s*xi[1], label='l=2')
plt.xscale('log')
plt.xlabel('$s$ [$\mathrm{{Mpc}}/h$]')
plt.ylabel(r'$s \cdot \xi_{\ell}(s)$ [$\mathrm{{Mpc}}/h$]')
plt.legend()
plt.show()

In [ ]:
def plot_xi_styled(s, xi):
    fig, ax = plt.subplots(figsize=FIG_SIZE, dpi=300, constrained_layout=True)
    ax.semilogx(s, s * xi[0], lw=LINE_WIDTH, color=BART_COLOR, label=r'$\ell=0$')
    ax.semilogx(s, s * xi[1], lw=LINE_WIDTH, ls='--', color=COLA_COLOR, label=r'$\ell=2$')
    ax.set_xlabel(r'$s\,[\mathrm{Mpc}/h]$', fontsize=LABEL_SIZE)
    ax.set_ylabel(r'$s \cdot \xi_{\ell}(s)\,[\mathrm{Mpc}/h]$', fontsize=LABEL_SIZE)
    ax.legend(fontsize=LEGEND_SIZE, frameon=False, handlelength=1.8, borderaxespad=0.35, labelspacing=0.35)
    style_axis(ax)
    plt.show()


plot_xi_styled(s, xi)

## LRG power spectrum (FFT, nbodykit)

In [ ]:
from nbodykit.lab import ArrayCatalog, FFTPower


def compute_lrg_pk(cat, boxsize, tracer='LRG', nmesh=256, dk=None, kmin=0.0, subtract_shotnoise=True):
    mask = cat['TRACER'] == tracer
    pos = np.column_stack([
        cat['x'][mask] % boxsize,
        cat['y'][mask] % boxsize,
        cat['z'][mask] % boxsize,
    ]).astype('f4')

    galaxy_cat = ArrayCatalog({'Position': pos})
    mesh = galaxy_cat.to_mesh(Nmesh=nmesh, BoxSize=boxsize, resampler='tsc',
                               interlaced=True, compensated=True)
    result = FFTPower(mesh, mode='1d', dk=dk or 2 * np.pi / boxsize, kmin=kmin)

    k = result.power['k']
    Pk = result.power['power'].real
    if subtract_shotnoise:
        Pk = Pk - result.power.attrs['shotnoise']

    valid = k > 0  # drop the k=0 DC bin
    return k[valid], Pk[valid]


def plot_lrg_pk_styled(k, Pk, label='LRG'):
    fig, ax = plt.subplots(figsize=FIG_SIZE, dpi=300, constrained_layout=True)
    ax.loglog(k, Pk, lw=LINE_WIDTH, color=BART_COLOR, label=label)
    ax.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=LABEL_SIZE)
    ax.set_ylabel(r'$P(k)\,[(\mathrm{Mpc}/h)^3]$', fontsize=LABEL_SIZE)
    ax.legend(fontsize=LEGEND_SIZE, frameon=False, handlelength=1.8, borderaxespad=0.35, labelspacing=0.35)
    style_axis(ax)
    plt.show()

In [ ]:
k_lrg, Pk_lrg = compute_lrg_pk(cat, HOD_obj.boxsize, tracer='LRG')
plot_lrg_pk_styled(k_lrg, Pk_lrg, label='LRG (this work)')